In [1]:
import cv2 
import urllib.request
import os
import time

In [4]:
url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
filename = "haarcascade_frontalface_default.xml"

if not os.path.exists(filename):
    print('downloading haar cascade file...')
    urllib.request.urlretrieve(url, filename)
    print('download completed')


save_path = "faces"
os.makedirs(save_path, exist_ok=True)


capture_delay = 5   
last_capture_time = 0
face_count = 0


frame_skip = 3
frame_id = 0


faceCascade = cv2.CascadeClassifier(filename)


video_path = "video.mp4"   
cap = cv2.VideoCapture(video_path)


def detect_faces(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = faceCascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,   
        minSize=(30, 30)
    )

    return faces


while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1


    if frame_id % frame_skip != 0:
        continue

    faces = detect_faces(frame)

    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x,y), (x+w, y+h), (255,0,0), 2)
        cv2.putText(frame, "Face", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2)

   
    if len(faces) > 0:
        current_time = time.time()

        if current_time - last_capture_time > capture_delay:
            for (x, y, w, h) in faces:
                face_crop = frame[y:y+h, x:x+w]

                face_count += 1
                file_name = os.path.join(save_path, f"face_{face_count}.jpg")

                cv2.imwrite(file_name, face_crop)
                print(f"Saved: {file_name}")

            last_capture_time = current_time

   
    cv2.putText(frame, f"Faces Saved: {face_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Face Detection", frame)

    if cv2.waitKey(1) == 27:
        break

cap.release()
cv2.destroyAllWindows()

Saved: faces\face_1.jpg
Saved: faces\face_2.jpg
Saved: faces\face_3.jpg
Saved: faces\face_4.jpg
Saved: faces\face_5.jpg
Saved: faces\face_6.jpg
